# EMA + Bollinger Backtest (Corrected)

This notebook implements your original Backtrader strategy with corrections:

- Removed duplicated historical-data fetch block
- Added robust numeric conversion and cleanup
- Fixed breakout logic to compare previous close with previous Bollinger band
- Uses environment variables instead of hardcoded credentials
- Exports generated signals to CSV

In [ ]:
# ==============================
# IMPORTS
# ==============================

import os
import datetime as dt

import backtrader as bt
import pandas as pd
from breeze_connect import BreezeConnect

In [ ]:
# ==============================
# BREEZE CONNECTION
# ==============================

API_KEY = os.getenv("BREEZE_API_KEY", "").strip()
API_SECRET = os.getenv("BREEZE_API_SECRET", "").strip()
SESSION_TOKEN = os.getenv("BREEZE_SESSION_TOKEN", "").strip()

if not API_KEY or not API_SECRET or not SESSION_TOKEN:
    raise ValueError(
        "Set BREEZE_API_KEY, BREEZE_API_SECRET, and BREEZE_SESSION_TOKEN environment variables."
    )

breeze = BreezeConnect(api_key=API_KEY)
breeze.generate_session(api_secret=API_SECRET, session_token=SESSION_TOKEN)

In [ ]:
# ==============================
# FETCH 5-MINUTE DATA (CLEANED)
# ==============================

FROM_DATE = "2026-01-02T09:15:00.000Z"
TO_DATE = dt.datetime.utcnow().strftime("%Y-%m-%dT%H:%M:%S.000Z")


def fetch_5m_data(client: BreezeConnect, stock_code: str, from_date: str, to_date: str) -> pd.DataFrame:
    data = client.get_historical_data(
        interval="5minute",
        from_date=from_date,
        to_date=to_date,
        stock_code=stock_code,
        exchange_code="NSE",
        product_type="cash",
    )

    rows = data.get("Success") or []
    if not rows:
        raise ValueError(f"No historical data returned for {stock_code}.")

    df_local = pd.DataFrame(rows)

    # Datetime parse
    df_local["datetime"] = pd.to_datetime(df_local["datetime"], errors="coerce")

    # Critical numeric conversion
    numeric_cols = ["open", "high", "low", "close", "volume"]
    for col in numeric_cols:
        df_local[col] = pd.to_numeric(df_local[col], errors="coerce")

    # Drop invalid rows and keep required columns
    df_local.dropna(subset=["datetime", "open", "high", "low", "close"], inplace=True)
    df_local.set_index("datetime", inplace=True)
    df_local.sort_index(inplace=True)

    return df_local[["open", "high", "low", "close", "volume"]]


# Example: NIFTY cash index bars
df = fetch_5m_data(
    client=breeze,
    stock_code="NIFTY",
    from_date=FROM_DATE,
    to_date=TO_DATE,
)

print(df.dtypes)
df.head()

In [ ]:
# ==============================
# BACKTRADER DATA FEED + STRATEGY
# ==============================

class CustomPandasData(bt.feeds.PandasData):
    params = (
        ("datetime", None),
        ("open", "open"),
        ("high", "high"),
        ("low", "low"),
        ("close", "close"),
        ("volume", "volume"),
        ("openinterest", -1),
    )


class FiveEmaBollStrategy(bt.Strategy):
    params = (
        ("ema_period", 5),
        ("bb_period", 20),
        ("bb_dev", 1.5),
    )

    def __init__(self):
        self.ema = bt.indicators.EMA(self.data.close, period=self.p.ema_period)
        self.boll = bt.indicators.BollingerBands(
            self.data.close,
            period=self.p.bb_period,
            devfactor=self.p.bb_dev,
        )

        self.bb_upper = self.boll.top
        self.bb_lower = self.boll.bot

        self.signals = []
        self.last_signal = None

    def next(self):
        min_len = max(self.p.ema_period, self.p.bb_period)
        if len(self.data) < (min_len + 1):
            return

        price = self.data.close[0]
        prev_price = self.data.close[-1]

        ema = self.ema[0]
        upper = self.bb_upper[0]
        lower = self.bb_lower[0]

        # Correction: use previous band values for crossover checks
        prev_upper = self.bb_upper[-1]
        prev_lower = self.bb_lower[-1]

        current_dt = self.data.datetime.datetime(0)

        # LONG: close crosses above upper band and is above EMA
        if (
            prev_price <= prev_upper
            and price > upper
            and price > ema
            and self.last_signal != "LONG"
        ):
            self.signals.append(
                {
                    "datetime": current_dt,
                    "signal": "LONG",
                    "price": float(price),
                    "ema": float(ema),
                    "bb_upper": float(upper),
                }
            )
            self.last_signal = "LONG"

        # SHORT: close crosses below lower band and is below EMA
        elif (
            prev_price >= prev_lower
            and price < lower
            and price < ema
            and self.last_signal != "SHORT"
        ):
            self.signals.append(
                {
                    "datetime": current_dt,
                    "signal": "SHORT",
                    "price": float(price),
                    "ema": float(ema),
                    "bb_lower": float(lower),
                }
            )
            self.last_signal = "SHORT"

In [ ]:
# ==============================
# BACKTEST ENGINE + ANALYZERS
# ==============================

cerebro = bt.Cerebro()

# Add strategy
cerebro.addstrategy(FiveEmaBollStrategy)

# Add data
data_feed = CustomPandasData(dataname=df)
cerebro.adddata(data_feed)

# Starting capital
cerebro.broker.set_cash(1_000_000)

# Analyzers
cerebro.addanalyzer(bt.analyzers.TradeAnalyzer, _name="trades")
cerebro.addanalyzer(bt.analyzers.SharpeRatio, _name="sharpe")
cerebro.addanalyzer(bt.analyzers.DrawDown, _name="drawdown")

# Run backtest
results = cerebro.run()
strat = results[0]

print("Final Portfolio Value:", cerebro.broker.getvalue())

print("\nTrade Analysis:")
print(strat.analyzers.trades.get_analysis())

print("\nSharpe Ratio:")
print(strat.analyzers.sharpe.get_analysis())

print("\nDrawdown:")
print(strat.analyzers.drawdown.get_analysis())

In [ ]:
# ==============================
# EXPORT SIGNALS + OPTIONAL PLOT
# ==============================

signals_df = pd.DataFrame(strat.signals)

if not signals_df.empty:
    signals_df.to_csv("ema_boll_signals.csv", index=False)
    print("Signals exported to ema_boll_signals.csv")
    display(signals_df.head())
else:
    print("No signals generated.")

# Plot (works best in local notebook GUI backends)
# If plotting fails in headless environments, comment this line.
cerebro.plot(style="candlestick")